# Phase 3b: self-distillation data (the base model's own correct answers)

**Round self2 (current).** In round self1 the base model answered the physics questions with the **official UGPhysics prompt**. It spent much of its thinking arguing about the answer format ("the user said ... (unit) ... not in the box"). 42% of its finished answers closed the thinking, wrote an answer, reopened it and closed it again. After filtering, only about 1% of physics answers were clean.

So now the model answers the physics questions with a **simple prompt** (the bare question plus "Please reason step by step, and put your final answer within \\boxed{}"). Its correct answers are then trained under the **official prompt**, with the final line rewritten to the official form ("So the final answer is \\boxed{...} unit."). The model learns to answer the official prompt with clean, focused reasoning instead of arguing about the format. The maths examples from self1 were fine and are reused.

**How the work is shared:** the questions are split into 3 shards. Each account answers one shard at the same time:
- Account A: `SHARD = 0`
- Account B: `SHARD = 1`
- Account C: `SHARD = 2`

**Before you start**
1. Replace `physself.py` with version 1.2. Keep `physeval.py` (1.3), `physdata.py` and `phystrain.py` in the same folder.
2. **Runtime > Change runtime type > T4 GPU** (step 3 and step 8 alone can run on a CPU runtime).

## 1. Settings

In [ ]:
WORKDIR = "/content/drive/MyDrive/physics_eval"
NAME = "self2"                 # this round (self1 = the first round with the official prompt)
SHARD = 0                      # account A: 0, account B: 1, account C: 2
NUM_SHARDS = 3
BACKEND = "vllm"               # worked in round self1 (about 1,290 tok/s per account)
SAMPLES = 2                    # answers per question (the shortest correct one is kept)
TEMPERATURE, TOP_P, TOP_K = 0.6, 0.95, 50
MAX_NEW = 3584                 # answers longer than this are not kept anyway
HF_BATCH = 64                  # only for BACKEND = "hf"
TIME_BUDGET_MIN = 0            # 0 = run until this shard is done; e.g. 150 to stop cleanly after 2.5 hours
MATH_SHARE = 0.0               # self2: physics only (the maths examples come from self1)
ADD_FROM = "self1"             # reuse these finished examples ...
ADD_STYLES = "math"            # ... of this style

## 2. Mount Drive and check files

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys
for f in ["physeval.py", "physdata.py", "physself.py", "train_data/v1.2/train.jsonl", "data/items_ugphysics.jsonl"]:
    assert os.path.exists(f"{WORKDIR}/{f}"), f"missing {f}"
sys.path.insert(0, WORKDIR)
import physeval; assert physeval.VERSION >= "1.3", "replace physeval.py with version 1.3"
print("ok")

## 3. Choose the questions (ONE account only, once, about 1 minute)
Takes 9,000 physics questions with checkable reference answers from data v1.2 (already leak-checked), with subjects in exam proportions. Each one stores two prompts: the simple one used for generating, and the official one used for training. **Run this on one account only**, before the others start step 6.

In [ ]:
!python "{WORKDIR}/physself.py" select --workdir "{WORKDIR}" --name {NAME} --source v1.2 --n 9000 --math-share {MATH_SHARE} --gen-prompt plain

## 4a. Install vLLM (only if `BACKEND = "vllm"`; 3 to 6 minutes)

In [ ]:
!pip -q install -U vllm
!pip -q install "antlr4-python3-runtime==4.11.1" chardet

## 4b. Install for transformers (only if `BACKEND = "hf"`)

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U transformers accelerate "antlr4-python3-runtime==4.11.1" chardet

## 5. Speed test (optional now: vLLM already ran at about 1,290 tok/s per account in round self1)
Skip it, or run it to confirm the setup works in this session.

In [ ]:
!python "{WORKDIR}/physself.py" generate --workdir "{WORKDIR}" --name {NAME} --shard {SHARD} --num-shards {NUM_SHARDS} \
    --backend {BACKEND} --samples {SAMPLES} --temperature {TEMPERATURE} --top-p {TOP_P} --top-k {TOP_K} \
    --max-new-tokens {MAX_NEW} --batch-size {HF_BATCH} --chunk 128 --limit 128

## 6. Generate this account's shard (the long step)
Saves after every chunk. **Safe to interrupt:** after a disconnect, run steps 1, 2, 4a (or 4b), then this cell again; it skips everything already done.

With 9,000 questions x 2 answers, each shard has 6,000 answers of up to 3,584 tokens. Your speed test tells you the time: hours = 6,000 x 2,500 / (tok/s x 3,600).

In [ ]:
!python "{WORKDIR}/physself.py" generate --workdir "{WORKDIR}" --name {NAME} --shard {SHARD} --num-shards {NUM_SHARDS} \
    --backend {BACKEND} --samples {SAMPLES} --temperature {TEMPERATURE} --top-p {TOP_P} --top-k {TOP_K} \
    --max-new-tokens {MAX_NEW} --batch-size {HF_BATCH} --time-budget-min {TIME_BUDGET_MIN}

## 7. Progress of all shards (any account)

In [ ]:
!python "{WORKDIR}/physself.py" status --workdir "{WORKDIR}" --name {NAME}

## 8. Build the training data (ONE account, when the shards are finished; CPU is fine, 10 to 30 minutes)
Grades every answer with the official UGPhysics judge, applies the quality checks, keeps the shortest clean correct answer per question, rewrites its last line to the official format, and adds the maths examples from self1. Writes `train_data/self2/` for the trainer (`DATA = "self2"`).

In [ ]:
!pip -q install "antlr4-python3-runtime==4.11.1" chardet scipy
!python "{WORKDIR}/physself.py" build --workdir "{WORKDIR}" --name {NAME} --out {NAME} --add-from {ADD_FROM} --add-styles {ADD_STYLES}
from IPython.display import Markdown, display
display(Markdown(open(f"{WORKDIR}/train_data/{NAME}/stats.md").read()))

## 9. Read a few kept answers

In [ ]:
display(Markdown(open(f"{WORKDIR}/train_data/{NAME}/samples.md").read()))